# A2 - Knowledge-Base Demo
Show OCR quality on a sample and one working retrieval example.

## OCR quality on a sample

The demo retrieval below indexes human-gold label text because the measured OCR baseline is not yet strong enough to seed a reliable full-corpus knowledge base.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

baseline_path = ROOT / "reports" / "ocr_baseline.json"
baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
summary = baseline["summary"]

print(f"model: {baseline['model']}")
print(
    f"pages: {summary['pages']}  mean F1: {summary['mean_f1']:.3f}  "
    f"median F1: {summary['median_f1']:.3f}  "
    f"range: {summary['min_f1']:.3f}-{summary['max_f1']:.3f}  "
    f"truncated: {summary['truncated_pages']}/{summary['pages']}"
)


## Retrieval demo

Read `grading_kit/labels.jsonl`, convert finished human-gold label rows to `Chunk` objects, chunk/embed/index them into FAISS, then run a natural Bangla question and assert that the expected page is retrieved.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import sys
from pathlib import Path

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(ROOT)

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

from doc_agent import config
from doc_agent.contracts import Chunk
from doc_agent.index import chunk as chunk_mod
from doc_agent.index import embed, store

cfg = config.load(ROOT / "configs" / "config.yaml")

labels_path = ROOT / "grading_kit" / "labels.jsonl"
gold: dict[str, str] = {}
with labels_path.open(encoding="utf-8") as fh:
    for line in fh:
        if not line.strip():
            continue
        record = json.loads(line)
        text = (record.get("text") or "").strip()
        if record.get("status") == "done" and text:
            gold[record["page_id"]] = text

groups: dict[str, str] = {}
groups_path = ROOT / "data" / "deed_groups.csv"
if groups_path.exists():
    with groups_path.open(encoding="utf-8", newline="") as fh:
        for row in csv.DictReader(fh):
            groups[row["page_id"]] = row["doc_id"]

source_chunks = [
    Chunk(
        id=f"{page_id}#gold",
        doc_id=groups.get(page_id, page_id),
        text=text,
        page_ids=[page_id],
    )
    for page_id, text in sorted(gold.items())
]

split_chunks = chunk_mod.split(source_chunks, cfg)
vectors = embed.encode(split_chunks, cfg)
store.build(split_chunks, vectors, cfg)

query = "দলিলে টেনু সাব্বির পিতার নাম কী?"
expected_page_id = "dolil_605"
k = 3

loaded = store.load(cfg)
query_vector = embed.encode_query(query, cfg)
scores, indices = loaded["index"].search(query_vector, k)

top_idx = int(indices[0][0])
top_score = float(scores[0][0])
top_record = loaded["metadata"][top_idx]

print(f"demo index built from {len(gold)} human-gold label pages")
print(f"chunks indexed: {len(split_chunks)}")
print(f"query: {query}")
print(f"expected_page_id: {expected_page_id}")
print(f"retrieved_page_id: {top_record['page_id']}")
print(f"chunk_id: {top_record['chunk_id']}")
print(f"score: {top_score:.6f}")
print("chunk:")
print(top_record["chunk_text"])

assert top_record["page_id"] == expected_page_id
